# Commercial Vehicle Data Platform

## 01 — DT-CARGO Data Exploration

Explore the DT-CARGO fleet and track datasets to understand their structure, data quality, and key relationships before building the data pipeline.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DT-CARGO Exploration")
    .getOrCreate()
)

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 18:08:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.0.1


## Fleet Data

In [3]:
fleet_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/workspace/spark/local/data/raw/fleet.csv")
)

fleet_df.printSchema()

root
 |-- vehicle_id: integer (nullable = true)
 |-- fleet_test_id: integer (nullable = true)
 |-- gross_vehicle_weight: double (nullable = true)
 |-- total_mass_with_trailer: double (nullable = true)
 |-- axle_class: double (nullable = true)



In [4]:
print(f"Row count: {fleet_df.count()}")

fleet_df.show(5, truncate=False)

Row count: 163
+----------+-------------+--------------------+-----------------------+----------+
|vehicle_id|fleet_test_id|gross_vehicle_weight|total_mass_with_trailer|axle_class|
+----------+-------------+--------------------+-----------------------+----------+
|1         |1            |18000.0             |40000.0                |98.0      |
|2         |1            |18000.0             |40000.0                |98.0      |
|3         |1            |18000.0             |40000.0                |98.0      |
|4         |1            |18000.0             |40000.0                |98.0      |
|5         |1            |18000.0             |40000.0                |98.0      |
+----------+-------------+--------------------+-----------------------+----------+
only showing top 5 rows


In [5]:
print(f"Null vehicle_id: {fleet_df.filter(fleet_df.vehicle_id.isNull()).count()}")
print(f"Distinct vehicle_id: {fleet_df.select('vehicle_id').distinct().count()}")

Null vehicle_id: 0
Distinct vehicle_id: 163


## Tracks Data

In [6]:
tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/workspace/spark/local/data/raw/tracks.csv")
)

tracks_df.printSchema()

root
 |-- track_id: integer (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- tour_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- stop_time: timestamp (nullable = true)
 |-- distance: double (nullable = true)
 |-- track_gap: double (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- max_speed: double (nullable = true)
 |-- n_signal_loss: double (nullable = true)
 |-- d_signal_loss: double (nullable = true)
 |-- r_signal_loss: double (nullable = true)
 |-- avg_hdop: double (nullable = true)
 |-- home_base: boolean (nullable = true)
 |-- long_haul: boolean (nullable = true)
 |-- rest_area: boolean (nullable = true)
 |-- service_area_fuel: boolean (nullable = true)
 |-- industrial_area: boolean (nullable = true)
 |-- cid: integer (nullable = true)



In [7]:
print(tracks_df.count())

tracks_df.show(5, truncate=False)

151598
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|track_id|vehicle_id|tour_id|start_time             |stop_time              |distance|track_gap  |avg_speed|max_speed|n_signal_loss|d_signal_loss|r_signal_loss|avg_hdop  |home_base|long_haul|rest_area|service_area_fuel|industrial_area|cid|
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|0       |1         |0      |2021-09-10 09:04:53.598|2021-09-10 10:07:57.395|30751.0 |NULL       |8.13     |20.58    |0.0          |0.0          |0.0          |0.6123537 |false    |false    |false    |false            |true           |149|
|1       |1         |0      |2021

In [8]:
track_vehicle_count = tracks_df.select("vehicle_id").distinct().count()

print(f"Distinct vehicles in tracks: {track_vehicle_count}")

Distinct vehicles in tracks: 163


## Speed Data

In [9]:
speed_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/workspace/spark/local/data/raw/speed/1/0.csv")
)

speed_df.printSchema()

root
 |-- epoch: double (nullable = true)
 |-- speed: double (nullable = true)
 |-- hdop: double (nullable = true)



In [8]:
print(f"Row count: {speed_df.count()}")

speed_df.show(10, truncate=False)

Row count: 37126
+----------------+-----+----+
|epoch           |speed|hdop|
+----------------+-----+----+
|1.631271893602E9|0.03 |0.8 |
|1.631271893704E9|0.01 |0.8 |
|1.631271893806E9|0.03 |0.8 |
|1.631271893908E9|0.02 |0.8 |
|1.63127189401E9 |0.03 |0.8 |
|1.631271894112E9|0.06 |0.8 |
|1.631271894214E9|0.04 |0.8 |
|1.631271894316E9|0.02 |0.8 |
|1.631271894418E9|0.02 |0.8 |
|1.63127189452E9 |0.03 |0.8 |
+----------------+-----+----+
only showing top 10 rows


## Raw Data Exploration — Summary

The DT-CARGO source data consists of three datasets:

- `fleet.csv` — vehicle-level information
- `tracks.csv` — track-level information
- `speed/` — speed observations for individual tracks

The speed files follow the structure `speed/{vehicle_id}/{track_id}.csv`.

The schemas, sample records, and basic dataset sizes were inspected to understand the source data before implementing the Bronze layer.

In [11]:
spark.stop()